In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
url = 'https://github.com/felvieira/claude-skills-fv/archive/refs/heads/main'
resp = requests.get(url)

#github.com/felvieira/claude-skills-fv/archive/refs/heads/main.zip

#<a class="prc-ActionList-ActionListContent-KBb8- prc-Link-Link-9ZwDx" tabindex="0" aria-labelledby="_r_94_--label" id="_r_94_" data-size="medium" data-turbo="false" href="" rel="nofollow"><span class="prc-ActionList-Spacer-4tR2m"></span><span class="prc-ActionList-LeadingVisual-NBr28 prc-ActionList-VisualWrap-bdCsS"><svg aria-hidden="true" focusable="false" class="octicon octicon-file-zip" viewBox="0 0 16 16" width="16" height="16" fill="currentColor" display="inline-block" overflow="visible" style="vertical-align: text-bottom;"><path d="M3.5 1.75v11.5c0 .09.048.173.126.217a.75.75 0 0 1-.752 1.298A1.748 1.748 0 0 1 2 13.25V1.75C2 .784 2.784 0 3.75 0h5.586c.464 0 .909.185 1.237.513l2.914 2.914c.329.328.513.773.513 1.237v8.586A1.75 1.75 0 0 1 12.25 15h-.5a.75.75 0 0 1 0-1.5h.5a.25.25 0 0 0 .25-.25V4.664a.25.25 0 0 0-.073-.177L9.513 1.573a.25.25 0 0 0-.177-.073H7.25a.75.75 0 0 1 0 1.5h-.5a.75.75 0 0 1 0-1.5h-3a.25.25 0 0 0-.25.25Zm3.75 8.75h.5c.966 0 1.75.784 1.75 1.75v3a.75.75 0 0 1-.75.75h-2.5a.75.75 0 0 1-.75-.75v-3c0-.966.784-1.75 1.75-1.75ZM6 5.25a.75.75 0 0 1 .75-.75h.5a.75.75 0 0 1 0 1.5h-.5A.75.75 0 0 1 6 5.25Zm.75 2.25h.5a.75.75 0 0 1 0 1.5h-.5a.75.75 0 0 1 0-1.5ZM8 6.75A.75.75 0 0 1 8.75 6h.5a.75.75 0 0 1 0 1.5h-.5A.75.75 0 0 1 8 6.75ZM8.75 3h.5a.75.75 0 0 1 0 1.5h-.5a.75.75 0 0 1 0-1.5ZM8 9.75A.75.75 0 0 1 8.75 9h.5a.75.75 0 0 1 0 1.5h-.5A.75.75 0 0 1 8 9.75Zm-1 2.5v2.25h1v-2.25a.25.25 0 0 0-.25-.25h-.5a.25.25 0 0 0-.25.25Z"></path></svg></span><span class="prc-ActionList-ActionListSubContent-gKsFp" data-component="ActionList.Item--DividerContainer"><span id="_r_94_--label" class="prc-ActionList-ItemLabel-81ohH">Download ZIP</span></span></a>

In [3]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    #prefix = 'https://github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                #print(post.metadata['title']) 
                #print(post.metadata['tags']) 
                data['filename'] = filename
                #print(data)
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    #print(repository_data)
    return repository_data

In [4]:
# function to split in chunks

def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [6]:
docs_ia = read_repo_data('felvieira', 'claude-skills-fv')
#evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(docs_ia)}")
#print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 154


In [9]:
# function to split in chunks

def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

docs_ia_chunks = []  # Initialize the list to store chunks

for doc in docs_ia:
    if 'content' not in doc:
        continue  # Skip documents that don't have a 'content' key to avoid KeyError
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    docs_ia_chunks.extend(chunks)  # Append to docs_ia_chunks instead of modifying docs_ia during iteration

In [8]:
docs_ia_chunks = []

for doc in docs_ia:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    docs_ia.extend(chunks)

KeyError: 'content'

In [34]:
doc_copy = docs_ia[3].copy()
doc_content = doc_copy.pop('content')


sliding_window(doc_content, 2000, 1000)

[{'start': 0,
  'chunk': '# Dev Team Kit for Coding Agents\n\n![Version](https://img.shields.io/badge/version-1.0.0-0f766e)\n![Skills](https://img.shields.io/badge/skills-28-1d4ed8)\n![Setup](https://img.shields.io/badge/setup-multi--platform-f59e0b)\n![Docs](https://img.shields.io/badge/docs-governed-7c3aed)\n\n> Um kit visual e governado para agentes de coding trabalharem com pipeline, contexto persistente, auditoria de repo e skills especializadas.\n\n## Visao Geral\n\nEste repositorio entrega um sistema completo para agentes compativeis com Claude, OpenCode, Copilot, Windsurf, Gemini CLI e Antigravity:\n\n- `GLOBAL.md` define as regras universais do kit\n- `policies/` concentra execucao, seguranca de tools, handoffs, evals e qualidade\n- `skills/*/SKILL.md` implementa 28 especialistas numerados\n- `templates/` oferece formatos curtos e reutilizaveis\n- `docs/` guarda guias, quickstart, contexto e auditorias reutilizaveis\n- `patterns/ai-integration/` organiza padroes para integrar 

In [28]:
docs_ia_chunks[550]

{'start': 0,
 'chunk': '# Release Plan\n- Escopo:\n- Versao/changelog:\n- Rollout:\n- Rollback:\n- Comunicacao:\n- Responsaveis:',
 'filename': 'claude-skills-fv-main/templates/release-plan.md'}

In [ ]:
#There are some alternative approaches:

#Token-based chunking: You first tokenize the content (turn it into a sequence of words) and then do a sliding window over tokens
#Advantages: More precise control over LLM input size
#Disadvantages: Doesn't work well for documents with code
#Paragraph splitting: Split by paragraphs
#Section splitting: Split by sections
#AI-powered splitting: Let AI split the text intelligently

#We won't cover token-based chunking here, as we're working with documents that contain code. But it's easy to implement - ask ChatGPT for help if you need it for text-only content.


In [36]:
# 2 splitting by paragraphs

import re
text = docs_ia[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())


#We use \n\s*\n regex pattern for splitting:

#\n matches a newline
#\s* matches zero or more whitespace characters
#\n matches another newline
#So \n\s*\n matches two newlines with optional whitespace between them

#This works well for literature, but it doesn't work well for documents. Most paragraphs in technical documentation are very short.

#You can combine sliding window and paragraph splitting for more intelligent processing. We won't do it here, but it's a good exercise to try.

#Let's now look at section splitting. Here, we take advantage of the documents' structure. Markdown documents have this structure:



## split by headers

import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


#If we want to split by second-level headers, that's what we do:

#sections = split_markdown_by_level(text, level=2)

docs_ia_chunks = []



for doc in docs_ia:
    if 'content' not in doc:
        continue  # Skip documents that don't have a 'content' key to avoid KeyError
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        docs_ia_chunks.append(section_doc)



In [37]:
print(docs_ia_chunks)

[{'filename': 'claude-skills-fv-main/AGENTS.md', 'section': '## Objetivo\n\nEste repositorio define um kit de skills e governanca para agentes de coding em Claude, OpenCode e superficies compativeis.'}, {'filename': 'claude-skills-fv-main/AGENTS.md', 'section': '## Uso em Repos Consumidores\n\n- em repos de aplicacao, o modo recomendado e manter `AGENTS.md` na raiz e instalar o kit dentro de `.bot/`\n- usar `templates/AGENTS-root.md` como base para o `AGENTS.md` do repo consumidor\n- se o repo consumidor nao tiver auditoria valida, iniciar por `Repo Auditor`'}, {'filename': 'claude-skills-fv-main/AGENTS.md', 'section': '## Ordem de Leitura\n\n1. `GLOBAL.md`\n2. `policies/`\n3. `README.md`\n4. `skills/*/SKILL.md`\n5. `docs/skill-guides/` somente quando a tarefa exigir exemplos extensos'}, {'filename': 'claude-skills-fv-main/AGENTS.md', 'section': '## Defaults Operacionais\n\n- responder curto por padrao\n- agir primeiro quando houver default seguro\n- nao repetir contexto desnecessario\

In [ ]:
#3. Intelligent Chunking with LLM

In [ ]:
# Let's stop Jupyter and create an environment variable with your key:

#export OPENAI_API_KEY='your-api-key'


In [ ]:
#Install the OpenAI SDK:

uv add openai

#Then run jupyter notebook:

uv run jupyter notebook

#It's cumbersome to set environment variables every time. I recommend using direnv, which works for Linux, Mac and Windows.

#Note: if you use direnv, don't forget to add .envrc to .gitignore.

#Warning: Never commit your API keys to git! Others can use your API key and you'll pay for it.

#Now we're ready to use OpenAI:

from openai import OpenAI

openai_client = OpenAI()


def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text

#This code invokes an LLM (gpt-4o-mini) with the provided prompt and returns the results. We will explain in more detail what this code does in the next lessons.

#Let's create a prompt:

prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

#The prompt asks the LLM to:

#Split the document logically (not just by length)
#Make sections self-contained
#Use a specific output format that's easy to parse

#Let's create a function for intelligent chunking:

def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

#Now we apply this to every document:

from tqdm.auto import tqdm

docs_ia_chunks = []

for doc in tqdm(docs_ia):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

#tqdm is a library that shows progress bars. It helps you track progress when processing a large number of documents.

#Note: This process requires time and incurs costs. As mentioned before, use this only when really necessary. For most applications, you don't need intelligent chunking.

#Bonus: you can use this approach for processing the code in your GitHub repository. You can use a variation of the following prompt:

#"Summarize the code in plain English. Briefly describe each class and function/method (their purpose and role), then give a short overall summary of how they work together. Avoid low-level details.". Then add both the source code and the summary to your documents.
#4. How to Choose a Chunking Approach
#You may wonder - which chunking should I use? The answer: start with the simplest one and gradually increase complexity. Start with simple chunking with overlaps. We will later talk about evaluations. You can use evaluations to make informed decisions about chunking strategies.


In [37]:
print(dtc_faq[153])

{'content': '# Risco\n- Tipo:\n- Impacto:\n- Probabilidade:\n- Mitigacao:\n- Dono:', 'filename': 'claude-skills-fv-main/templates/risk-note.md'}
